In [10]:
import argparse
import os
import jax
import torch 
import numpy as np
import pickle
from quantum_transformers.datasets import get_custom_classification_dataloader
from quantum_transformers.transformers import Transformer
from quantum_transformers.quantum_layer_qiskit import (
    get_circuit, 
    basic_vqc, 
    angle_embedding,
    phase_embedding,
    amplitude_embedding,
    iqp_embedding
)
from quantum_transformers.training import train_and_evaluate
from typing import Callable, Tuple
import tensorcircuit as tc
import jax.numpy as jnp
import flax.linen as nn

In [11]:
print("JAX devices:", jax.devices())

JAX devices: [CudaDevice(id=0)]


In [12]:
HIDDEN_SIZE = 8      
NUM_HEADS = 2
NUM_BLOCKS = 2
MLP_HIDDEN = 8             
MAX_SEQ_LEN = 32
BATCH_SIZE = 16   ## try to increase again
NUM_EPOCHS = 40       
LEARNING_RATE = 1e-3
NUM_LAYERS_VQC = 2

DATA_PATHS = {
    'mc': {
        'type': 'mc_rp',
        'train': 'data/mc_train_data.txt',
        'val': 'data/mc_dev_data.txt',
        'test': 'data/mc_test_data.txt',
        'tokenizer': 'wordlevel',
        'vocab': 17,  
        'shared_files': None
    },
    'rp': {
        'type': 'mc_rp',
        'train': 'data/rp_train_data.txt',
        'val': None,
        'test': 'data/rp_test_data.txt',
        'tokenizer': 'wordlevel',
        'vocab': 115, 
        'shared_files': None
    }
}

SENTIMENT_FILES = [
    'data/imdb_labelled.txt', 
    'data/amazon_cells_labelled.txt', 
    'data/yelp_labelled.txt'
]

In [13]:
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


In [14]:
dataset_config = DATA_PATHS['mc']

train_loader, val_loader, test_loader, tokenizer = get_custom_classification_dataloader(
                dataset_type='mc_rp',
                train_path=dataset_config['train'],
                val_path=dataset_config['val'],
                test_path=dataset_config['test'],
                batch_size=BATCH_SIZE, # Use dynamic batch size
                max_seq_len=MAX_SEQ_LEN,
                tokenizer_type=dataset_config['tokenizer'], 
                vocab_size=dataset_config['vocab'],         
                tokenizer_files=None             
            ) 

Data Loaded. Train: 70, Val: 30, Test: 30
Tokenizer (wordlevel) trained. Vocab size: 19


In [15]:
circuit_fn = get_circuit(
    embedding=angle_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)



In [16]:
model_0 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

In [17]:
print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_0,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)

Starting Training...


ValueError: vmap got inconsistent sizes for array axes to be mapped:
  * one axis had size 512: axis 0 of argument inputs of type float32[512,8];
  * one axis had size 2: axis 0 of argument weights of type float32[2,8]

In [ ]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=angle_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

     ┌───────┐┌───────┐                                                       »
q_0: ┤ Ry(0) ├┤ Rx(0) ├──■────────────────────────────────────────────────────»
     ├───────┤├───────┤┌─┴─┐     ┌───────┐                                    »
q_1: ┤ Ry(0) ├┤ Rx(0) ├┤ X ├──■──┤ Rx(0) ├────────────────────────────────────»
     ├───────┤├───────┤└───┘┌─┴─┐└───────┘┌───────┐                           »
q_2: ┤ Ry(0) ├┤ Rx(0) ├─────┤ X ├────■────┤ Rx(0) ├───────────────────────────»
     ├───────┤├───────┤     └───┘  ┌─┴─┐  └───────┘┌───────┐                  »
q_3: ┤ Ry(0) ├┤ Rx(0) ├────────────┤ X ├──────■────┤ Rx(0) ├──────────────────»
     ├───────┤├───────┤            └───┘    ┌─┴─┐  └───────┘┌───────┐         »
q_4: ┤ Ry(0) ├┤ Rx(0) ├─────────────────────┤ X ├──────■────┤ Rx(0) ├─────────»
     ├───────┤├───────┤                     └───┘    ┌─┴─┐  └───────┘┌───────┐»
q_5: ┤ Ry(0) ├┤ Rx(0) ├──────────────────────────────┤ X ├──────■────┤ Rx(0) ├»
     ├───────┤├───────┤                 

## Phase Embedding

In [ ]:
# from quantum_transformers.quantum_layer import phase_embedding

circuit_fn = get_circuit(
    embedding=phase_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)

model_1 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_1,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)



Starting Training...
Starting training for 40 epochs (Seed: 0)...


Epoch 1 | Train Loss: 0.7007 | Val Loss: 0.8052, Val AUC: 0.7560


Epoch 2 | Train Loss: 0.6696 | Val Loss: 0.7639, Val AUC: 0.7608


Epoch 3 | Train Loss: 0.6791 | Val Loss: 0.7527, Val AUC: 0.7847


Epoch 4 | Train Loss: 0.6681 | Val Loss: 0.7408, Val AUC: 0.7895


...

Epoch 37 | Train Loss: 0.6570 | Val Loss: 0.6584, Val AUC: 0.9952


Epoch 38 | Train Loss: 0.6477 | Val Loss: 0.6626, Val AUC: 0.9952


Epoch 39 | Train Loss: 0.6532 | Val Loss: 0.6683, Val AUC: 0.9952


Epoch 40 | Train Loss: 0.6346 | Val Loss: 0.6680, Val AUC: 0.9952
Total training time = 93.81s, Best AUC = 99.52% at epoch 35


Test Loss = 0.6588, Test AUC = 95.56%


In [ ]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
# dummy_inputs = jnp.zeros((2,))
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
# dummy_weights = jnp.zeros((NUM_LAYERS_VQC, 2))
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=phase_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

     ┌───┐┌───────┐┌───┐┌───────┐                                              »
q_0: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├──■───────────────────────────────────────────»
     ├───┤├───────┤├───┤├───────┤┌─┴─┐     ┌───────┐                           »
q_1: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├┤ X ├──■──┤ Rx(0) ├───────────────────────────»
     ├───┤├───────┤├───┤├───────┤└───┘┌─┴─┐└───────┘┌───────┐                  »
q_2: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├─────┤ X ├────■────┤ Rx(0) ├──────────────────»
     ├───┤├───────┤├───┤├───────┤     └───┘  ┌─┴─┐  └───────┘┌───────┐         »
q_3: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├────────────┤ X ├──────■────┤ Rx(0) ├─────────»
     ├───┤├───────┤├───┤├───────┤            └───┘    ┌─┴─┐  └───────┘┌───────┐»
q_4: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├─────────────────────┤ X ├──────■────┤ Rx(0) ├»
     ├───┤├───────┤├───┤├───────┤                     └───┘    ┌─┴─┐  └───────┘»
q_5: ┤ H ├┤ Rz(0) ├┤ H ├┤ Rx(0) ├──────────────────────────────┤ X ├──────■────»
     ├───┤├───────┤├───┤├───

# Amplitude Embedding

In [ ]:
# from quantum_transformers.quantum_layer import phase_embedding

circuit_fn = get_circuit(
    embedding=amplitude_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)

model_2 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_2,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)



Starting Training...


AttributeError: 'Circuit' object has no attribute 'initialize'

In [ ]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
# dummy_inputs = jnp.zeros((2,))
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
# dummy_weights = jnp.zeros((NUM_LAYERS_VQC, 2))
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=amplitude_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

# IQP Embedding

In [ ]:
# from quantum_transformers.quantum_layer import phase_embedding

circuit_fn = get_circuit(
    embedding=iqp_embedding,
    vqc=basic_vqc
)
current_w_shape = (NUM_LAYERS_VQC,)

model_3 = Transformer(
    num_tokens=tokenizer.get_vocab_size(),
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    hidden_size=HIDDEN_SIZE,
    num_heads=NUM_HEADS,
    num_transformer_blocks=NUM_BLOCKS,
    mlp_hidden_size=MLP_HIDDEN,
    dropout=0.1,
    quantum_w_shape=current_w_shape,
    quantum_attn_circuit=circuit_fn,
    quantum_mlp_circuit=circuit_fn
)

print("Starting Training...")
trial_num = globals().get("trial_num", 0)

(test_loss, test_acc), best_state, history = train_and_evaluate(
    model=model_3,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    task='classification',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    seed=trial_num
)



Starting Training...
Starting training for 40 epochs (Seed: 0)...


Epoch 1 | Train Loss: 0.6817 | Val Loss: 0.6958, Val AUC: 0.7656


Epoch 2 | Train Loss: 0.6744 | Val Loss: 0.7084, Val AUC: 0.7751


Epoch 3 | Train Loss: 0.6843 | Val Loss: 0.7192, Val AUC: 0.7943


Epoch 4 | Train Loss: 0.6791 | Val Loss: 0.7185, Val AUC: 0.8182


...

Epoch 37 | Train Loss: 0.6209 | Val Loss: 0.6648, Val AUC: 0.9904


Epoch 38 | Train Loss: 0.6288 | Val Loss: 0.6649, Val AUC: 0.9904


Epoch 39 | Train Loss: 0.6198 | Val Loss: 0.6685, Val AUC: 0.9904


Epoch 40 | Train Loss: 0.6317 | Val Loss: 0.6722, Val AUC: 0.9904
Total training time = 224.11s, Best AUC = 99.04% at epoch 33


Test Loss = 0.6563, Test AUC = 97.78%


In [ ]:
from quantum_transformers.quantum_layer import get_quantum_layer_circuit
import jax.numpy as jnp

# Create dummy inputs and weights based on the current configuration
# HIDDEN_SIZE is the number of qubits (8)
# dummy_inputs = jnp.zeros((2,))
dummy_inputs = jnp.zeros((HIDDEN_SIZE,))

# w_shape for basic_vqc is (NUM_LAYERS_VQC, HIDDEN_SIZE)
# dummy_weights = jnp.zeros((NUM_LAYERS_VQC, 2))
dummy_weights = jnp.zeros((NUM_LAYERS_VQC, HIDDEN_SIZE))

# Get the TensorCircuit Circuit object
c = get_quantum_layer_circuit(
    inputs=dummy_inputs, 
    weights=dummy_weights, 
    embedding=iqp_embedding, 
    vqc=basic_vqc
)

# Draw the circuit (outputs ASCII text representation by default)
print(c.draw())

     ┌───┐┌───────┐                                                    »
q_0: ┤ H ├┤ Rz(0) ├──■─────────────■────■─────────────■────■───────────»
     ├───┤├───────┤┌─┴─┐┌───────┐┌─┴─┐  │             │    │           »
q_1: ┤ H ├┤ Rz(0) ├┤ X ├┤ Rz(0) ├┤ X ├──┼─────────────┼────┼──────■────»
     ├───┤├───────┤└───┘└───────┘└───┘┌─┴─┐┌───────┐┌─┴─┐  │    ┌─┴─┐  »
q_2: ┤ H ├┤ Rz(0) ├───────────────────┤ X ├┤ Rz(0) ├┤ X ├──┼────┤ X ├──»
     ├───┤├───────┤                   └───┘└───────┘└───┘┌─┴─┐┌─┴───┴─┐»
q_3: ┤ H ├┤ Rz(0) ├──────────────────────────────────────┤ X ├┤ Rz(0) ├»
     ├───┤├───────┤                                      └───┘└───────┘»
q_4: ┤ H ├┤ Rz(0) ├────────────────────────────────────────────────────»
     ├───┤├───────┤                                                    »
q_5: ┤ H ├┤ Rz(0) ├────────────────────────────────────────────────────»
     ├───┤├───────┤                                                    »
q_6: ┤ H ├┤ Rz(0) ├────────────────────────────────